# Corrected Timestamp-Batch Sequence Parity Audit

This CPU-only audit fixes the legacy no-backfill sequence contract before any retraining. It independently compares:

1. a Spark timestamp-batch builder that removes the entire current timestamp before applying the 30/10 limits; and
2. an event-by-event online reducer that answers queries before appending current-time events.

Only a small report is saved. The full corrected Gold dataset is **not** materialized yet. Use a **High-RAM CPU** runtime; GPU is unnecessary.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_corrected_sequence_audit')
SOURCE_ROOT = WORK_ROOT / 'runtime'
REPORT_DIR = WORK_ROOT / 'report'
SPARK_TMP = Path('/content/spark-corrected-sequence')
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/corrected_sequence_audit_v1'
for path in (SOURCE_ROOT, REPORT_DIR, DRIVE_REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('canonical history:', BASE_GOLD_ROOT / 'history/events')
print('locked protocol:', PROTOCOL_DIR)

In [ ]:
required = {
    'pyspark': 'pyspark==3.5.6',
    'pyarrow': 'pyarrow',
    'pandas': 'pandas',
    'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

required_paths = [
    BASE_GOLD_ROOT / 'history/events',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    PROTOCOL_DIR / 'queries.parquet',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing required inputs:\n' + '\n'.join(missing_paths)
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= 20, f'At least 20 GiB local disk is required; only {free_gib:.1f} GiB is free.'
print(f'preflight: PASS | local disk free: {free_gib:.1f} GiB')

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILENAME = 'runtime/corrected_sequence_audit_runtime_v1.zip'
RUNTIME_EXPECTED_SHA256 = '80bd0904488305b4666f30aae6c39bc405cfb18966928637a8b8dc7cf46cd491'

def normalize_hf_token(raw):
    if isinstance(raw, bytes):
        raw = raw.decode('utf-8')
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_hf_token(value)
            if token:
                return token
    if isinstance(raw, (list, tuple)):
        for value in raw:
            token = normalize_hf_token(value)
            if token:
                return token
    return None

try:
    HF_TOKEN = normalize_hf_token(userdata.get('HF_TOKEN'))
except Exception as exc:
    print('Colab Secret unavailable:', type(exc).__name__)
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = normalize_hf_token(getpass('Paste Hugging Face READ token: '))
assert HF_TOKEN, 'HF_TOKEN must begin with hf_.'

runtime_zip = Path(hf_hub_download(
    repo_id=HF_REPO_ID, filename=RUNTIME_FILENAME, repo_type='model', token=HF_TOKEN,
    force_download=True,
))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
AUDIT_SCRIPT = SOURCE_ROOT / 'scripts/audit_corrected_adaptive_sequences.py'
assert AUDIT_SCRIPT.exists()
compile(AUDIT_SCRIPT.read_text(), str(AUDIT_SCRIPT), 'exec')
print('runtime integrity: PASS')

## Run full corrected parity audit

Spark will scan canonical history, construct timestamp batches, then run the independent per-user event reducer. Intermediate shuffle data stays on local Colab disk. Progress can pause during Spark window/shuffle stages.

In [ ]:
for path in (REPORT_DIR, SPARK_TMP):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)

command = [
    sys.executable, str(AUDIT_SCRIPT),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--protocol-dir', str(PROTOCOL_DIR),
    '--output-dir', str(REPORT_DIR),
    '--max-sequence-len', '30',
    '--max-session-sequence-len', '10',
    '--calibration-user-fraction', '0.35',
    '--seed', '44',
    '--shuffle-partitions', '32',
]
env = os.environ.copy()
env.update({
    'SPARK_MASTER': 'local[2]',
    'PYSPARK_SUBMIT_ARGS': '--driver-memory 8g pyspark-shell',
    'SPARK_LOCAL_DIR': str(SPARK_TMP),
    'PYTHONPATH': str(SOURCE_ROOT) + os.pathsep + env.get('PYTHONPATH', ''),
})
log_path = Path('/content/corrected_sequence_audit.log')
process = subprocess.Popen(
    command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
with log_path.open('w') as log:
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-240:])
    raise RuntimeError(f'Corrected sequence audit failed with exit code {return_code}:\n{tail}')
required_outputs = [
    REPORT_DIR / 'corrected_sequence_audit.json',
    REPORT_DIR / 'corrected_sequence_parity_per_query.csv',
    REPORT_DIR / 'corrected_sequence_mismatch_examples.json',
]
assert all(path.exists() for path in required_outputs), 'Audit did not produce all reports.'
print('audit artifacts: PASS')

In [ ]:
report = json.loads((REPORT_DIR / 'corrected_sequence_audit.json').read_text())
print(json.dumps(report, indent=2))
parity = report['corrected_batch_vs_event_reducer']
print('\nACCEPTED:', parity['accepted'])
print('corrected parity:', f"{parity['core_fields_match_rate']:.4%}")
print('legacy count match:', f"{report['legacy_vs_corrected']['recent_count_match_rate']:.4%}")

In [ ]:
bundle_base = Path('/content/corrected_sequence_audit_v1')
bundle_path = Path(shutil.make_archive(str(bundle_base), 'zip', REPORT_DIR))
drive_bundle = DRIVE_REPORT_DIR / bundle_path.name
shutil.copy2(bundle_path, drive_bundle)
print('Drive backup:', drive_bundle, f'{drive_bundle.stat().st_size / 1024**2:.2f} MiB')
files.download(str(bundle_path))

## Acceptance rule

Proceed to a full adaptive Gold rebuild and V4 retraining only when `corrected_batch_vs_event_reducer.accepted` is `true`. If it is false, inspect the mismatch examples first; do not train another model on an unresolved state contract.